In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
base_url = "http://books.toscrape.com/catalogue/page-{}.html"
main_url = "http://books.toscrape.com/catalogue/"

data = []

for page in range(1, 51):
    url = base_url.format(page)
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    books = soup.find_all("article", class_="product_pod")

    for book in books:
        title = book.h3.a["title"]
        price = book.find("p", class_="price_color").text
        rating = book.p["class"][1]

        # Get product link
        relative_link = book.h3.a["href"]
        product_link = main_url + relative_link

        # Visit product page
        product_response = requests.get(product_link)
        product_soup = BeautifulSoup(product_response.text, "html.parser")

        # Extract additional details
        try:
            description = product_soup.find("meta", attrs={"name": "description"})["content"].strip()
        except:
            description = "No description"

        try:
            category = product_soup.find("ul", class_="breadcrumb").find_all("li")[2].text.strip()
        except:
            category = "Unknown"

        table = product_soup.find("table", class_="table table-striped")
        rows = table.find_all("tr")

        product_info = {}
        for row in rows:
            key = row.th.text
            value = row.td.text
            product_info[key] = value

        data.append({
            "title": title,
            "price": price,
            "rating": rating,
            "availability": product_info.get("Availability"),
            "UPC": product_info.get("UPC"),
            "product_type": product_info.get("Product Type"),
            "tax": product_info.get("Tax"),
            "num_reviews": product_info.get("Number of reviews"),
            "category": category,
            "description": description
        })

In [4]:
df = pd.DataFrame(data)
df.to_csv("cleaned_data.csv", index=False)

print("Done!")

Done!
